=============================================================
#  CodeAlpha Internship — Task 3: Music Generation with AI
# Developer Name: Tauseef Alam
#  Platform  : Google Colab  
#  Dataset   : Bach Chorales (built into music21 — no download!)
#  Model     : Bidirectional LSTM (deep learning)
#  Libraries : music21, tensorflow, gradio, fluidsynth
================================================================

In [1]:
# 1. Install system-level audio dependencies for MIDI to WAV conversion
!apt-get update -qq
!apt-get install -y -qq fluidsynth fluid-soundfont-gm

# 2. Install Python libraries without forcing conflicting versions
!pip install -q music21 gradio pretty_midi

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libdouble-conversion3:amd64.
(Reading database ... 122363 files and directories currently installed.)
Preparing to unpack .../00-libdouble-conversion3_3.1.7-4_amd64.deb ...
Unpacking libdouble-conversion3:amd64 (3.1.7-4) ...
Selecting previously unselected package libqt5core5a:amd64.
Preparing to unpack .../01-libqt5core5a_5.15.3+dfsg-2ubuntu0.2_amd64.deb ...
Unpacking libqt5core5a:amd64 (5.15.3+dfsg-2ubuntu0.2) ...
Selecting previously unselected package libevdev2:amd64.
Preparing to unpack .../02-libevdev2_1.12.1+dfsg-1_amd64.deb ...
Unpacking libevdev2:amd64 (1.12.1+dfsg-1) ...
Selecting previously unselected package libmtdev1:amd64.
Preparing to unpack .../03-libmtdev1_1.1.6-1build4_amd64.deb ...
Unpacking libmtdev1:

In [ ]:
import os
import random
import warnings
import subprocess
import numpy as np
import music21
import tensorflow as tf
import gradio as gr

# Suppress warnings and logs for a cleaner output
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

class AIMusicGenerator:
    def __init__(self):
        # Configuration
        self.sequence_length = 100
        self.vocab_size = 0
        self.note_to_int = {}
        self.int_to_note = {}
        self.network_input = []
        self.model = None

        # Paths
        self.model_path = "/content/best_music_model.keras"
        self.soundfont_path = "/usr/share/sounds/sf2/FluidR3_GM.sf2"

        # Setup reproducibility
        tf.random.set_seed(42)
        np.random.seed(42)

    def load_data(self, num_chorales=20):
        print(f"🎵 Loading {num_chorales} Bach chorales...")
        notes = []
        chorales = music21.corpus.getComposer('bach')[:num_chorales]

        for file in chorales:
            try:
                midi = music21.converter.parse(file)
                parts = music21.instrument.partitionByInstrument(midi)
                elements = parts.parts[0].recurse() if parts else midi.flat.notes

                for element in elements:
                    if isinstance(element, music21.note.Note):
                        notes.append(str(element.pitch))
                    elif isinstance(element, music21.chord.Chord):
                        notes.append('.'.join(str(n) for n in element.normalOrder))
            except Exception:
                continue # Skip unreadable files gracefully

        # Create Vocabulary
        unique_notes = sorted(set(notes))
        self.vocab_size = len(unique_notes)
        self.note_to_int = {note: i for i, note in enumerate(unique_notes)}
        self.int_to_note = {i: note for i, note in enumerate(unique_notes)}

        # Prepare Sequences
        network_input_raw = []
        network_output_raw = []

        for i in range(0, len(notes) - self.sequence_length):
            seq_in = notes[i:i + self.sequence_length]
            seq_out = notes[i + self.sequence_length]
            network_input_raw.append([self.note_to_int[char] for char in seq_in])
            network_output_raw.append(self.note_to_int[seq_out])

        self.network_input = np.reshape(network_input_raw, (len(network_input_raw), self.sequence_length))

        X = self.network_input / float(self.vocab_size)
        y = tf.keras.utils.to_categorical(network_output_raw, num_classes=self.vocab_size)

        print(f"✅ Data loaded! Unique notes: {self.vocab_size}, Training sequences: {len(X)}")
        return X, y

    def build_model(self):
        print("🧠 Building Bidirectional LSTM Model...")
        self.model = tf.keras.Sequential([
            tf.keras.layers.Embedding(self.vocab_size, 128, input_length=self.sequence_length),
            tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(256, return_sequences=True)),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(256)),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dense(256, activation='relu'),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(self.vocab_size, activation='softmax')
        ])

        self.model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
        print("✅ Model built successfully.")

    def train(self, X, y, epochs=40, batch_size=64): # Reduced epochs for faster initial testing
        print("🚀 Starting Training Process...")
        callbacks = [
            tf.keras.callbacks.ModelCheckpoint(self.model_path, monitor='loss', save_best_only=True, verbose=1),
            tf.keras.callbacks.EarlyStopping(monitor='loss', patience=8, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-6)
        ]

        self.model.fit(X, y, epochs=epochs, batch_size=batch_size, callbacks=callbacks)
        print(f"✅ Training complete. Best model saved to {self.model_path}")

    def generate_music(self, num_notes, temperature=1.0):
        if self.model is None:
            raise ValueError("Model must be trained before generating music.")

        start = np.random.randint(0, len(self.network_input) - 1)
        pattern = list(self.network_input[start])
        prediction_output = []

        for _ in range(num_notes):
            pred_input = np.reshape(pattern, (1, len(pattern))) / float(self.vocab_size)
            prediction = self.model.predict(pred_input, verbose=0)[0]

            # Temperature sampling for creativity
            prediction = np.log(prediction + 1e-7) / temperature
            exp_preds = np.exp(prediction)
            prediction = exp_preds / np.sum(exp_preds)
            index = np.argmax(np.random.multinomial(1, prediction, 1))

            prediction_output.append(self.int_to_note[index])
            pattern.append(index)
            pattern = pattern[1:]

        return prediction_output

    def export_audio(self, generated_notes):
        midi_path = "/content/generated_music.mid"
        wav_path = "/content/generated_music.wav"

        # 1. Notes to MIDI
        offset = 0
        output_notes = []
        for pattern in generated_notes:
            if ('.' in pattern) or pattern.isdigit():
                notes_in_chord = pattern.split('.')
                notes = []
                for current_note in notes_in_chord:
                    new_note = music21.note.Note(int(current_note))
                    new_note.storedInstrument = music21.instrument.Piano()
                    notes.append(new_note)
                new_chord = music21.chord.Chord(notes)
                new_chord.offset = offset
                output_notes.append(new_chord)
            else:
                new_note = music21.note.Note(pattern)
                new_note.offset = offset
                new_note.storedInstrument = music21.instrument.Piano()
                output_notes.append(new_note)
            offset += 0.5

        midi_stream = music21.stream.Stream(output_notes)
        midi_stream.write('midi', fp=midi_path)

        # 2. MIDI to WAV via FluidSynth Subprocess
        cmd = ["fluidsynth", "-ni", self.soundfont_path, midi_path, "-F", wav_path, "-r", "44100"]
        subprocess.run(cmd, capture_output=True)

        return midi_path, wav_path

# --- Initialization and UI ---
generator = AIMusicGenerator()

def run_training():
    X, y = generator.load_data()
    generator.build_model()
    generator.train(X, y)
    return "Training Complete! You can now generate music."

def generate_ui(num_notes, temp):
    if generator.model is None:
        return None, None, "⚠️ Please run the training step first!"
    notes = generator.generate_music(num_notes, temperature=temp)
    midi_file, wav_file = generator.export_audio(notes)
    return midi_file, wav_file, "✅ Music generated successfully!"

# Build the Gradio Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎹 Professional AI Music Generator")
    gr.Markdown("Uses a Bidirectional LSTM to learn Bach Chorales and synthesize new MIDI/WAV audio.")

    with gr.Row():
        with gr.Column():
            train_btn = gr.Button("1. Train Model (Run this first)", variant="primary")
            train_status = gr.Textbox(label="Training Status", interactive=False)

        with gr.Column():
            num_notes = gr.Slider(50, 300, value=100, step=10, label="Number of Notes")
            temperature = gr.Slider(0.1, 2.0, value=1.0, step=0.1, label="Creativity (Temperature)")
            gen_btn = gr.Button("2. Generate Music", variant="secondary")

    with gr.Row():
        audio_out = gr.Audio(label="Generated Audio (WAV)")
        file_out = gr.File(label="Download MIDI")
        gen_status = gr.Textbox(label="Generation Status", interactive=False)

    # Event Mapping
    train_btn.click(fn=run_training, outputs=train_status)
    gen_btn.click(fn=generate_ui, inputs=[num_notes, temperature], outputs=[file_out, audio_out, gen_status])

# Launch
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d711f4812b1ef1653a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🎵 Loading 20 Bach chorales...
✅ Data loaded! Unique notes: 54, Training sequences: 2958
🧠 Building Bidirectional LSTM Model...
✅ Model built successfully.
🚀 Starting Training Process...
Epoch 1/40
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.0385 - loss: 3.8930
Epoch 1: loss improved from None to 3.77312, saving model to /content/best_music_model.keras

Epoch 1: finished saving model to /content/best_music_model.keras
47/47 ━━━━━━━━━━━━━━━━━━━━ 13s 57ms/step - accuracy: 0.0497 - loss: 3.7731 - learning_rate: 0.0010
Epoch 2/40
46/47 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.0516 - loss: 3.6001
Epoch 2: loss improved from 3.77312 to 3.57795, saving model to /content/best_music_model.keras

Epoch 2: finished saving model to /content/best_music_model.keras
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - accuracy: 0.0571 - loss: 3.5780 - learning_rate: 0.0010
Epoch 3/40
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.0544 - loss: 3.5498
Epoch 3: loss improved from 3.57795 to 